# Program 20
## CBOW Model using Neural Networks
Implement Continuous Bag-of-Words (CBOW) for a sample sentence using basic Neural Network concepts.

### CBOW Architecture:
```
[Context Word 1] --\                  /-- [Hidden Layer] --\
[Context Word 2] ----> [Average/Sum] ----> [Output Layer] ---> [Target Word]
[Context Word N] --/                                       /
```

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Lambda
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# 1. Prepare Data
text = "the quick brown fox jumps over the lazy dog"

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
vocab_size = len(tokenizer.word_index) + 1
sequences = tokenizer.texts_to_sequences([text])[0]

# 2. Create input-output pairs (CBOW: Context -> Target)
window_size = 2
context_data = []
target_data = []

for i in range(window_size, len(sequences) - window_size):
    context = sequences[i - window_size : i] + sequences[i + 1 : i + window_size + 1]
    target = sequences[i]
    context_data.append(context)
    target_data.append(target)

X = np.array(context_data)
y = to_categorical(target_data, num_classes=vocab_size)

print("Sample X (contexts):", X[:3])
print("Sample y (targets):", np.argmax(y[:3], axis=1))

# 3. Define CBOW model architecture
embedding_dim = 10

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=window_size*2),
    # CBOW averages the context embeddings
    Lambda(lambda x: tf.reduce_mean(x, axis=1), output_shape=(embedding_dim,)),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# 4. Train the model
print("Training the CBOW model...")
model.fit(X, y, epochs=100, verbose=0)

# 5. Evaluate Performance
loss, acc = model.evaluate(X, y, verbose=0)
print(f"\nTraining Accuracy: {acc*100:.2f}%")

# Verify Output
word2idx = tokenizer.word_index
idx2word = {v: k for k, v in word2idx.items()}

def predict_center_word(context_words):
    context_idxs = [word2idx[w] for w in context_words]
    X_test = np.array([context_idxs])
    pred = model.predict(X_test, verbose=0)
    best_idx = np.argmax(pred)
    return idx2word[best_idx]

context_words = ['quick', 'brown', 'jumps', 'over']
pred_word = predict_center_word(context_words)
print(f"Context: {context_words} -> Predicted Target: {pred_word}")

Sample X (contexts): [[1 2 4 5]
 [2 3 5 6]
 [3 4 6 1]]
Sample y (targets): [3 4 5]


c:\Users\vishn\Desktop\Programs\SEM 8\NLP LAB\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Training the CBOW model...

Training Accuracy: 60.00%
Context: ['quick', 'brown', 'jumps', 'over'] -> Predicted Target: brown
